# RUNG-26b — de novo binder by **RFdiffusion** (v2: NO conda — pip, one Python, no restart)

v1 used condacolab, which swapped the Python and corrupted Boltz/ColabDesign — scrapped. v2 installs
RFdiffusion the **maintained ColabDesign way** (pip: dgl + e3nn + SE3Transformer), so Boltz, RFdiffusion,
ProteinMPNN and AF2 all live in ONE Python — no restart, no env-var loss, no corruption.

Pipeline: Boltz folds the target pMHC (proven) → RFdiffusion backbones to the mutated-residue hotspot →
ProteinMPNN sequences → AF2 (ColabDesign) scores vs MUT + WT.

**Run on a FRESH runtime** (Runtime ▸ Disconnect and delete runtime first — the old session is conda-poisoned).
Order: 1 clone → 2 Boltz fold (~20 min) → 3 install (~10-15 min, no restart) → 4 SMOKE (paste me this) → 5 batch (~4 h) → 6 rank.
Still GPU-fragile and untested-by-Claude, but this is the path the maintainers actually run. Target: IDH1-R132H/HLA-A\*01:01.

In [ ]:
#@title Cell 1 — clone + GPU + spec
import os, json, subprocess
from pathlib import Path
TARGET_ID = "IDH1_R132H_A0101" #@param {type:"string"}
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L
SPEC=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec',TARGET_ID],capture_output=True,text=True).stdout)
globals().update(GROOVE=SPEC['mhc_groove'], PEP_MUT=SPEC['pep_mut'], PEP_WT=SPEC['pep_wt'], MUTPOS=SPEC['mut_residue_in_peptide'], TARGET_ID=TARGET_ID)
print('target:',SPEC['gene'],SPEC['mut_label'],SPEC['allele'],'| mut',PEP_MUT,'WT',PEP_WT,'| mut pep pos',MUTPOS)
print('[CELL 1] OK')

In [ ]:
#@title Cell 2 — fold MUT/WT pMHC with BOLTZ -> PDB on Drive (proven)  [~20 min, GPU]
import os, glob, subprocess, shutil, importlib.util as iu, json
from google.colab import drive; drive.mount('/content/drive')
WORK='/content/drive/MyDrive/cancer-recon/rung26b_rfdiff'; os.makedirs(WORK, exist_ok=True)
if iu.find_spec('boltz') is None: subprocess.run(['pip','install','-q','boltz'])
if iu.find_spec('gemmi') is None: subprocess.run(['pip','install','-q','gemmi'])
subprocess.run(['pip','install','-q','numpy==1.26.4'])  # numba (in boltz) needs numpy<=2.1; fresh Colab ships 2.2
import gemmi
CACHE='/content/boltz_cache'; os.makedirs(CACHE, exist_ok=True)  # LOCAL (fresh per runtime) — a stale Drive cache caused 'CCD ALA not found'
def fold_pmhc(kind, pep):
    pdb=f'{WORK}/pmhc_{kind}.pdb'
    if os.path.exists(pdb): print(f'  {kind}: cached'); return pdb
    y=f'{WORK}/pmhc_{kind}.yaml'; outd=f'{WORK}/boltz_{kind}'
    open(y,'w').write('sequences:\n  - protein:\n      id: A\n      sequence: '+GROOVE+'\n  - protein:\n      id: B\n      sequence: '+pep+'\n')
    r=subprocess.run(f'boltz predict {y} --use_msa_server --diffusion_samples 1 --cache {CACHE} --out_dir {outd}', shell=True, capture_output=True, text=True)
    print((r.stderr or '')[-900:])
    f=sorted(glob.glob(f'{outd}/**/*.pdb',recursive=True)) or sorted(glob.glob(f'{outd}/**/*.cif',recursive=True))
    if not f: print(f'  {kind}: NO STRUCTURE'); return None
    if f[0].endswith('.cif'): st=gemmi.read_structure(f[0]); st.setup_entities(); st.write_pdb(pdb)
    else: shutil.copy(f[0], pdb)
    return pdb
MUT_PDB=fold_pmhc('mut',PEP_MUT); WT_PDB=fold_pmhc('wt',PEP_WT)
if MUT_PDB and WT_PDB:
    st=gemmi.read_structure(MUT_PDB); chB=[c for c in st[0] if c.name=='B'][0]; HOT=chB[MUTPOS-1].seqid.num
    json.dump({'mut_pdb':MUT_PDB,'wt_pdb':WT_PDB,'hotspot':HOT,'groove_len':len(GROOVE),'pep_len':len(PEP_MUT),'work':WORK,'target_id':TARGET_ID},
              open(f'{WORK}/meta.json','w'))
    print(f'[CELL 2] OK  hotspot=B{HOT}  meta.json written (all later cells read from it — restart-proof)')
else: print('[CELL 2] FAILED')

In [ ]:
#@title Cell 3 — install RFdiffusion (pip, NO conda) + ProteinMPNN + ColabDesign + params  [~12 min, NO restart]
import os, subprocess, glob
def sh(c): 
    r=subprocess.run(c, shell=True, capture_output=True, text=True); 
    if r.returncode: print('  ! ', c[:70], '\n', (r.stderr or '')[-400:])
    return r.returncode
if not os.path.exists('/content/RFdiffusion'):
    sh("apt-get -qq install -y aria2 >/dev/null")
    sh("git clone -q https://github.com/sokrypton/RFdiffusion.git /content/RFdiffusion")
    sh("git clone -q https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN")
    sh("pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    sh("pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    sh("pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    sh("pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx")
    sh("cd /content/RFdiffusion/env/SE3Transformer && pip install -q .")
    sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
    os.makedirs('/content/RFdiffusion/models', exist_ok=True)
    sh("cd /content/RFdiffusion/models && aria2c -q -x16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt")
    sh("cd /content/RFdiffusion/models && aria2c -q -x16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt")
if not glob.glob('params/params_model_*.npz'):
    sh("mkdir -p params && cd params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar) && tar -xf alphafold_params_2022-12-06.tar && rm -f alphafold_params_2022-12-06.tar")
sh('pip install -q numpy==1.26.4')  # lock the shared numpy after all installs (numba/boltz/colabdesign)
ri=glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py')
print('run_inference.py:', ri[0] if ri else 'NOT FOUND (paste Cell 3 output to Claude)')
print('Complex ckpt:', bool(glob.glob('/content/RFdiffusion/models/Complex_base_ckpt.pt')))
print('[CELL 3] done — if run_inference.py + ckpt present, proceed to Cell 4 SMOKE')

In [ ]:
#@title Cell 4 — SMOKE: 1 design end-to-end (RFdiffusion -> ProteinMPNN -> AF2 vs MUT+WT)  [~6-10 min, GPU]
import os, glob, subprocess, json
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
WORK=M['work']; MUT_PDB=M['mut_pdb']; WT_PDB=M['wt_pdb']; HOT=f"B{M['hotspot']}"; GL=M['groove_len']; PL=M['pep_len']
RI=(glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py'))[0]
CONTIG=f"[A1-{GL}/0 B1-{PL}/0 70-90]"      # keep MHC(A)+peptide(B) fixed; design a 70-90aa binder
def rfdiffuse(prefix, n):
    cmd=(f"cd /content/RFdiffusion && python {RI} inference.input_pdb={MUT_PDB} "
         f"'contigmap.contigs={CONTIG}' 'ppi.hotspot_res=[{HOT}]' "
         f"inference.output_prefix={prefix} inference.num_designs={n} inference.ckpt_override_path=models/Complex_base_ckpt.pt")
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True); print('  rf stderr:',(r.stderr or '')[-700:]); return r
def mpnn(bb):
    out=f'{WORK}/mpnn'; os.makedirs(out, exist_ok=True)
    subprocess.run(f"cd /content/ProteinMPNN && python protein_mpnn_run.py --pdb_path {bb} --pdb_path_chains B "
                   f"--out_folder {out} --num_seq_per_target 1 --sampling_temp 0.1 --seed 1 --batch_size 1",
                   shell=True, capture_output=True, text=True)
    fa=sorted(glob.glob(f'{out}/seqs/*.fa'))
    if not fa: return None
    s=[l.strip() for l in open(fa[-1]) if l.strip() and not l.startswith('>')][-1]
    return s.split('/')[-1] if '/' in s else s
from colabdesign import mk_afdesign_model, clear_mem
def score(target_pdb, seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder', use_multimer=False, num_recycles=3, data_dir='.')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=len(seq), hotspot=HOT, rm_target_seq=False, ignore_missing=True)
    m.predict(seq=seq, verbose=False); log=m.aux['log']
    return {'pae_interaction':round(float(log['i_pae'])*31.0,3),'binder_plddt':round(float(log['plddt'])*100.0,1)}
rfdiffuse(f'{WORK}/smoke/d', 1)
bb=sorted(glob.glob(f'{WORK}/smoke/*.pdb'))
print('backbone:', bb[-1] if bb else 'NONE — paste rf stderr above')
if bb:
    seq=mpnn(bb[-1]); print('MPNN seq:', seq)
    if seq:
        mut=score(MUT_PDB,seq); wt=score(WT_PDB,seq)
        print('MUT',mut,'| WT',wt,'| disc(Å)',round(wt['pae_interaction']-mut['pae_interaction'],2))
        print('[CELL 4] SMOKE OK — launch Cell 5')

In [ ]:
#@title Cell 5 — THE BATCH (time-boxed + resumable)  [~4 h, GPU]
max_hours=4.0 #@param {type:'number'}
import os, glob, time, json
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); WORK=M['work']
DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True); PAE_BIND=10.0; PLDDT_MIN=80.0
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/metrics.json'))
print(f'[CELL 5] resume from {i}; until', time.strftime('%H:%M',time.localtime(t_end)))
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/metrics.json'): i+=1; continue
    os.makedirs(dd, exist_ok=True)
    try:
        rfdiffuse(f'{dd}/bb', 1); bb=sorted(glob.glob(f'{dd}/*.pdb'))
        if not bb: raise RuntimeError('no backbone')
        seq=mpnn(bb[-1])
        if not seq: raise RuntimeError('no mpnn seq')
        mut=score(MUT_PDB,seq); rec={'design_id':did,'target':M['target_id'],'sequence':seq,'mut':mut}
        if mut['pae_interaction']<=PAE_BIND and mut['binder_plddt']>=PLDDT_MIN: rec['wt']=score(WT_PDB,seq)
        json.dump(rec, open(f'{dd}/metrics.json','w'))
        print(f"  {did}: mut_pae={mut['pae_interaction']} plddt={mut['binder_plddt']}"+(f" DISC={round(rec['wt']['pae_interaction']-mut['pae_interaction'],2)}" if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/metrics.json','w'))
    i+=1
print(f'[CELL 5] done — {i} attempted. Run Cell 6.')

In [ ]:
#@title Cell 6 — RANK + bundle
import os, sys, json, glob, zipfile, shutil
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); DES=f"{M['work']}/designs"
dst='runs/rung26b_rfdiff'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m,d)
os.system(f'{sys.executable} scripts/52_binder_design.py rank {dst}')
b='/content/rung26b_rfdiff.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True):
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')